In [ ]:
from langchain.chains import LLMChain
import os

In [ ]:
from langchain.prompts import PromptTemplate

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

True

In [ ]:
# !pip install --upgrade langchain-google-genai

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))
llm.invoke("Write me a ballad about LangChain")

AIMessage(content="In digital realms, where circuits gleam,\nAnd data flows in endless stream,\nA legend rose, a guiding light,\nTo tame the chaos, day and night.\nHis name was LangChain, strong and bold,\nA story waiting to unfold.\n\nFrom scattered thoughts and queries vast,\nA tapestry of knowledge cast,\nHe saw the need for order's hand,\nTo make the AI understand.\nNo longer lost in foggy haze,\nBut guided through intelligent maze.\n\nHe built his chains, both long and deep,\nWhere models whispered secrets keep.\nEach link a tool, a function bright,\nTo fetch, to parse, to shed new light.\nFrom simple words to complex lore,\nHe opened up a brand new door.\n\nThe LLMs, with power grand,\nOnce roamed untamed across the land.\nBut LangChain brought them to their knees,\nWith structured thought and wise decrees.\nHe showed them how to reason clear,\nAnd banish doubt, and conquer fear.\n\nHe gathered knowledge, far and wide,\nFrom databases, where truths reside.\nHe learned to speak, t

In [ ]:
prompt = PromptTemplate(
    input_variables=["product"],
    template="What is a good name for a company that makes {product} only suggest max 5 names?")

In [ ]:
chain = LLMChain(llm=llm, prompt=prompt, output_key="response")


In [ ]:
out = chain.invoke({"product": "colorful socks"})

In [ ]:
print(out["response"])

Here are 5 name suggestions for a colorful sock company:

1.  **Chroma Threads**
2.  **Vivid Strides**
3.  **Spectrum Socks**
4.  **Hue Haven**
5.  **KaleidoKicks**


## Runnable

In [ ]:
chain = prompt | llm

out = chain.invoke({"product": "colorful socks"})

print(out.content)

Here are 5 name suggestions for a colorful sock company:

1.  **KaleidoSox** (Combines "kaleidoscope" with "socks" for a vibrant, patterned feel)
2.  **ChromaThreads** ( "Chroma" refers to color, and "Threads" implies the material of socks)
3.  **PrismPeds** ( "Prism" evokes bright, scattered colors, and "Peds" is a common slang for socks)
4.  **HueHaven** (Suggests a place where all the colors of socks can be found)
5.  **VividFeet** (Simple, direct, and emphasizes the bright, lively nature of the socks)


## Runnable Passthrough

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Create a RunnablePassthrough instance
runnable = RunnablePassthrough()

# Pass data through without modification
result = runnable.invoke({"key": "value"})
print(result)

{'key': 'value'}


### Sequential Flow and Chain

In [ ]:
# Define a prompt template
template = "Explain {concept} in simple terms."
prompt = PromptTemplate(template=template, input_variables=["concept"])
user_input = "quantum computing"
# Format and send to LLM
formatted_prompt = prompt.format(concept=user_input)
result = llm.invoke(formatted_prompt)

In [ ]:
from datetime import datetime

# Process LLM response
draft = {
    "explanation": result.content,
    "timestamp": datetime.now().isoformat()
}

In [ ]:
summarise_template = "Summarize the following in less than {max_words} words: {text}"
summarize_prompt = PromptTemplate(
    input_variables=["text", "max_words"],
    template=summarise_template
)

# create summarize chain using LCEL composition (prompt | llm)
summarize_chain = summarize_prompt | llm

# Summarize the draft explanation
final_output = summarize_chain.invoke({
    "text": draft["explanation"],
    "max_words": 50
})

# Return the summarized content
final_summary = {"summary": final_output.content}
final_summary

{'summary': 'Quantum computers use qubits, which can be 0, 1, or both simultaneously (superposition). This, along with entanglement, allows them to explore many solutions at once, making them vastly more powerful for complex problems like drug discovery and optimization.'}

In [ ]:
# Build two LLMChains and compose them into a SequentialChain:
# - First chain explains the concept and returns its output under the key "text"
# - Second chain summarizes the "text" using the existing summarize_prompt
from langchain.chains import SequentialChain

explain_chain_seq = LLMChain(llm=llm, prompt=prompt, output_key="text")
summarize_chain_seq = LLMChain(llm=llm, prompt=summarize_prompt, output_key="summary")

seq_chain = SequentialChain(
    chains=[explain_chain_seq, summarize_chain_seq],
    input_variables=["concept", "max_words"],
    output_variables=["text", "summary"],
    verbose=True
)

# Invoke the sequential chain using the existing user_input
result = seq_chain.invoke({"concept": user_input, "max_words": 50})

# Print both the full explanation and the concise summary
print("Explanation:\n", result["text"], "\n")
print("Summary:\n", result["summary"])



> Entering new SequentialChain chain...

> Finished chain.
Explanation:
 Imagine a regular computer is like a light switch. It can be either **on** (representing a 1) or **off** (representing a 0). This is called a **bit**. All the information in your computer is stored as a long string of these 0s and 1s.

Quantum computers are different. Instead of just being on or off, their fundamental unit, called a **qubit**, can be **on, off, or somewhere in between**. Think of it like a dimmer switch. It can be fully on, fully off, or at any level of brightness in between. This "somewhere in between" is called **superposition**.

This ability to be in multiple states at once is a game-changer. Here's why:

**1. Superposition: Doing Many Things at Once**

Because a qubit can be in superposition, a quantum computer with just a few qubits can represent a vast number of possibilities simultaneously.

*   **Regular Computer:** If you have 2 bits, you can represent 4 possible combinations (00, 01, 

## Output Parser

In [ ]:
import json
import re
from typing import Any

# Simple JSON output parser example for use with existing `llm` and `PromptTemplate`

class SimpleJSONOutputParser:
    """Very small parser that extracts the first JSON object/array from text and returns it as Python data."""
    def parse(self, text: str) -> Any:
        # find first {...} or [...] block (simple heuristics)
        m = re.search(r'(\{.*\}|\[.*\])', text, re.S)
        json_text = m.group(1) if m else text
        try:
            return json.loads(json_text)
        except Exception as e:
            raise ValueError(f"Could not parse JSON: {e}\nOriginal text: {text}")

    def format_instructions(self) -> str:
        # helpful when including parser instructions in the prompt
        return "Return valid JSON only (an object or array)."

# Example prompt that asks the LLM to return JSON
json_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Describe {topic} in a short JSON object with keys 'title' and 'description'. {instructions}"
)

# Format with parser instructions and call the LLM directly (using existing `llm`)
parser = SimpleJSONOutputParser()
formatted = json_prompt.format(topic="quantum computing", instructions=parser.format_instructions())

# Call the LLM and parse the response
raw_response = llm.invoke(formatted)  # returns an AIMessage-like object
parsed_json = parser.parse(raw_response.content)

print("Parsed JSON:", parsed_json)

Parsed JSON: {'title': 'Quantum Computing', 'description': 'Quantum computing leverages quantum mechanical phenomena like superposition and entanglement to perform computations. Unlike classical bits (0 or 1), quantum bits (qubits) can represent 0, 1, or a combination of both simultaneously, enabling them to solve certain complex problems exponentially faster than classical computers.'}


In [ ]:
parsed_json

{'title': 'Quantum Computing',
 'description': 'Quantum computing leverages quantum mechanical phenomena like superposition and entanglement to perform computations. Unlike classical bits (0 or 1), quantum bits (qubits) can represent 0, 1, or a combination of both simultaneously, enabling them to solve certain complex problems exponentially faster than classical computers.'}

## Chat Prompt Template

In [ ]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
# System message template
system_template = "You are an assistant that helps with {task}."
system_message = SystemMessagePromptTemplate.from_template(system_template)

In [ ]:
# Human message template
human_template = "{user_query}"
human_message = HumanMessagePromptTemplate.from_template(human_template)

In [ ]:
from langchain.prompts import MessagesPlaceholder
# Compose full chat template with history
chat_prompt = ChatPromptTemplate.from_messages([
system_message,
human_message
])

In [ ]:
llm.invoke(chat_prompt.format_messages(task="programming", user_query="How do I write a for loop in Python?"))

AIMessage(content='You\'re asking a fundamental question in Python! For loops are incredibly useful for iterating over sequences of items. Here\'s a breakdown of how to write them, along with various examples:\n\n## The Basic Structure of a `for` Loop\n\nThe general syntax for a `for` loop in Python is:\n\n```python\nfor item in iterable:\n    # Code to be executed for each item\n    # This block is indented\n```\n\nLet\'s break down the components:\n\n*   **`for`**: This keyword signifies the start of a for loop.\n*   **`item`**: This is a variable that will take on the value of each element in the `iterable` one by one during each iteration of the loop. You can name this variable anything you like, but it\'s good practice to choose a name that describes the items being iterated over (e.g., `number` for numbers, `char` for characters, `name` for names).\n*   **`in`**: This keyword separates the `item` variable from the `iterable`.\n*   **`iterable`**: This is any Python object that ca

## Exercise 1

In [ ]:
"""
Exercise (fill in the blanks with your implementation):

Goal:
- Take an Amazon review string.
- Determine sentiment: "positive", "negative", or "neutral".
- If sentiment == "neutral", produce an apologetic response draft for the customer.

Hints-only: major parts are left for students to implement. Use the existing `llm`,
`PromptTemplate` / chat templates, and the SimpleJSONOutputParser (defined earlier)
to help produce and parse structured LLM responses.
"""

# Part 1: Sample input (student should replace with other examples for testing)
amazon_review = "..."  # HINT: assign a realistic Amazon review string here

# Part 2: Sentiment analysis function (student implements)
def analyze_sentiment(review: str) -> str:
    """
    HINTS:
    - Use the LLM to classify sentiment. Ask the model to return a tiny JSON
      object like: {"sentiment": "positive"} where sentiment is one of
      "positive", "negative", or "neutral".
    - You can build a PromptTemplate that includes an instruction to return valid JSON only.
    - Use an output parser (e.g., SimpleJSONOutputParser.parse) to convert the LLM text to a dict,
      then return the value for the "sentiment" key.
    - Validate/paranoid-check the parser output before returning (fallback to "neutral" if unclear).
    """
    # TODO: implement using llm, PromptTemplate, and JSON parsing
    raise NotImplementedError("Fill in analyze_sentiment using the LLM and a JSON output parser.")


# Part 3: Apology generator (student implements)
def generate_apology(review: str) -> str:
    """
    HINTS:
    - This only runs when sentiment == "neutral".
    - Write a prompt that asks the assistant to generate a short apologetic reply:
       - Acknowledge the customer's experience.
       - Express empathy and apologize.
       - Offer a next step (refund, replacement, contact support).
       - Keep it concise (1-3 sentences).
    - You can use a ChatPromptTemplate with a SystemMessage for tone ("You are a helpful customer support agent...")
      and a HumanMessage with the review and the expected output constraints.
    - Invoke the existing `llm` with the formatted prompt and return llm response text (or .content).
    """
    # TODO: implement using chat prompt + llm
    raise NotImplementedError("Fill in generate_apology using the LLM to produce an apologetic reply.")


# Part 4: Main flow (students wire pieces together)
def main(review: str):
    # HINT: call analyze_sentiment(review)
    # - If "positive": print a short acknowledgment (no apology needed).
    # - If "negative": suggest an empathetic escalation message or steps for follow-up.
    # - If "neutral": call generate_apology(review) and print the apology text.
    # Also print the detected sentiment for clarity.
    sentiment = None  # TODO: replace with analyze_sentiment(review)
    apology = None

    if sentiment == "positive":
        print({"sentiment": sentiment, "message": "Thanks for the positive review!"})
    elif sentiment == "negative":
        # HINT: students could craft a suggested response or next-steps message here
        print({"sentiment": sentiment, "message": "Flag for customer support follow-up."})
    elif sentiment == "neutral":
        # HINT: call generate_apology and include the apology in the printed result
        apology = None  # TODO: replace with generate_apology(review)
        print({"sentiment": sentiment, "apology": apology})
    else:
        print({"sentiment": "unknown", "message": "Could not determine sentiment."})


# Quick test harness (students can modify)
if __name__ == "__main__":
    # HINT: Replace amazon_review above, then run this cell to test.
    main(amazon_review)

{'sentiment': 'unknown', 'message': 'Could not determine sentiment.'}


## Exercise 2

In [ ]:
# Part 1: User Question
# This is where the user's initial query is stored.
# HINT: Assign a sample user query string to the variable.
user_question = "..."

# Part 2: ChatPromptTemplate
# HINT: Create a list of dictionaries to represent the chat messages.
# The structure should be {"messages": [...]}.
# Each message in the list should have a "role" and "content".
# For example: {"role": "user", "content": user_question}
chat_messages = {
    "messages": [
        # Fill in the message details here
    ]
}

# Part 3: Chat Model
# HINT: This is a placeholder for a function that would interact with a real LLM.
# For this exercise, simulate the LLM's response.
# The response should be a string, and for the next step, it needs to contain
# information that can be parsed as JSON.
# Example response: '{"animal": "dog", "breed": "golden retriever"}'
def get_llm_response(messages):
    # Simulate LLM processing
    return "..."  # Your simulated response goes here

raw_response = get_llm_response(chat_messages)

# Part 4: OutputParser
# HINT: Use the `json` library to parse the raw_response string into a Python dictionary.
# Remember to handle potential errors if the string is not valid JSON.
import json

parsed_json = {}
try:
    # Your code to parse the JSON goes here
    pass
except json.JSONDecodeError:
    print("Error: The response from the model was not valid JSON.")

# Part 5: Next Chain
# HINT: This step processes the parsed_json.
# Add a new key to the dictionary or modify an existing one.
# For example, you could add a "status" field or a "processed_text" field.
processed_data = parsed_json
# Your processing code goes here

# Part 6: Final Answer
# HINT: Construct the final output for the user.
# The result should be a dictionary with an "answer" key.
# Use an f-string to format a user-friendly answer from the processed data.
final_answer = {
    "answer": "..."  # Your final formatted answer goes here
}

# Print the final result
print(final_answer)